In [2]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth

# 加载凭据文件
with open(expanduser('C:/Users/lixue/Desktop/worldquant/brain.txt')) as f:
    credentials = json.load(f)

# 从列表中提取用户名和密码
username, password = credentials

# 创建会话对象
sess = requests.Session()

# 设置基本身份验证
sess.auth = HTTPBasicAuth(username, password)

# 向API发送POST请求进行身份验证
response = sess.post('https://api.worldquantbrain.com/authentication')

# 打印响应状态和内容以调试
print(response.status_code)
print(response.json())

201
{'user': {'id': 'XL46509'}, 'token': {'expiry': 14400.0}, 'permissions': ['REFERRAL']}


In [3]:
# 获取数据集ID为fundamental6（Company Fundamental Data for Equity）下的所有数据字段
### Get Data_fields like Data Explorer 获取所有满足条件的数据字段及其ID
def get_datafields(
        s,
        searchScope,
        dataset_id: str = '',
        search: str = ''
):
    import pandas as pd
    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']

    if len(search) == 0:
        url_template = "https://api.worldquantbrain.com/data-fields?" + \
                       f"&instrumentType={instrument_type}" + \
                       f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50" + \
                       "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = "https://api.worldquantbrain.com/data-fields?" + \
                       f"&instrumentType={instrument_type}" + \
                       f"&region={region}&delay={str(delay)}&universe={universe}&limit=50" + \
                       f"&search={search}" + \
                       "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])

    datafields_list_flat = [item for sublist in datafields_list for item in sublist]

    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

# 爬取id
searchScope = {'region': 'USA', 'delay': '1', 'universe': 'TOP3000', 'instrumentType': 'EQUITY'}
fundamental6 = get_datafields(s=sess, searchScope=searchScope, dataset_id='fundamental6') # id设置

# 筛选（这里是type的MATRIX）
fundamental6 = fundamental6[fundamental6['type'] == "MATRIX"]
fundamental6.head()

datafields_list_fundamental6 = fundamental6['id'].values
print(datafields_list_fundamental6)
print(len(datafields_list_fundamental6))

['assets' 'assets_curr' 'bookvalue_ps' 'capex' 'cash' 'cash_st' 'cashflow'
 'cashflow_dividends' 'cashflow_fin' 'cashflow_invst' 'cashflow_op' 'cogs'
 'current_ratio' 'debt' 'debt_lt' 'debt_st' 'depre_amort' 'ebit' 'ebitda'
 'employee' 'enterprise_value' 'eps' 'equity' 'fnd6_acdo' 'fnd6_acodo'
 'fnd6_acox' 'fnd6_acqgdwl' 'fnd6_acqintan' 'fnd6_adesinda_curcd'
 'fnd6_aldo' 'fnd6_am' 'fnd6_aodo' 'fnd6_aox' 'fnd6_aqc' 'fnd6_aqi'
 'fnd6_aqs' 'fnd6_beta' 'fnd6_capxv' 'fnd6_ceql' 'fnd6_ch' 'fnd6_ci'
 'fnd6_cibegni' 'fnd6_cicurr' 'fnd6_cidergl' 'fnd6_cik' 'fnd6_cimii'
 'fnd6_ciother' 'fnd6_cipen' 'fnd6_cisecgl' 'fnd6_citotal' 'fnd6_city'
 'fnd6_cld2' 'fnd6_cld3' 'fnd6_cld4' 'fnd6_cld5' 'fnd6_cptmfmq_actq'
 'fnd6_cptmfmq_atq' 'fnd6_cptmfmq_ceqq' 'fnd6_cptmfmq_dlttq'
 'fnd6_cptmfmq_dpq' 'fnd6_cptmfmq_lctq' 'fnd6_cptmfmq_oibdpq'
 'fnd6_cptmfmq_opepsq' 'fnd6_cptmfmq_saleq' 'fnd6_cptnewqv1300_actq'
 'fnd6_cptnewqv1300_apq' 'fnd6_cptnewqv1300_atq' 'fnd6_cptnewqv1300_ceqq'
 'fnd6_cptnewqv1300_dlttq' 

In [4]:
# 将datafield替换到Alpha模板(框架)中group_rank({fundamental model data}/cap,subindustry)批量生成Alpha
alpha_list = []

for index,datafield in enumerate(datafields_list_fundamental6,start=1):
    print(f"正在循环第 {index} 个元素")
    print("正在将如下alpha表达式与setting封装")
    alpha_expression = f'group_rank(({datafield})/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)'
    print(alpha_expression)
    simulation_data = {
        "type": "REGULAR",
        "settings": {
            "instrumentType": "EQUITY",
            "region": "USA",
            "universe": "TOP3000",
            "delay": 1,
            "decay": 0,
            "neutralization": "SUBINDUSTRY",
            "truncation": 0.08,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "ON",
            "language": "FASTEXPR",
            "visualization": False,
        },
        "regular": alpha_expression
    }
    alpha_list.append(simulation_data)
    print(f"there are {len(alpha_list)} Alphas to simulate")

print(alpha_list[1])

正在循环第 1 个元素
正在将如下alpha表达式与setting封装
group_rank((assets)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there are 1 Alphas to simulate
正在循环第 2 个元素
正在将如下alpha表达式与setting封装
group_rank((assets_curr)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there are 2 Alphas to simulate
正在循环第 3 个元素
正在将如下alpha表达式与setting封装
group_rank((bookvalue_ps)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there are 3 Alphas to simulate
正在循环第 4 个元素
正在将如下alpha表达式与setting封装
group_rank((capex)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there are 4 Alphas to simulate
正在循环第 5 个元素
正在将如下alpha表达式与setting封装
group_rank((cash)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there are 5 Alphas to simulate
正在循环第 6 个元素
正在将如下alpha表达式与setting封装
group_rank((cash_st)/cap, subindustry)+ hump((ts_zscore(-returns,28))*rank(vwap/adv20), hump = 0.0012)
there ar

In [5]:
# 将Alpha一个一个发送至服务器进行回测（已经测试前两个了）
from time import sleep

for index,alpha in enumerate(alpha_list,start=1):
    sim_resp = sess.post(
        'https://api.worldquantbrain.com/simulations',
        json=alpha,
    )

    try:
        sim_progress_url = sim_resp.headers['Location']
        while True:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = float(sim_progress_resp.headers.get("Retry-After", 0))
            if retry_after_sec == 0:  # simulation done!模拟完成!
                break
            sleep(retry_after_sec)
        alpha_id = sim_progress_resp.json()["alpha"]  # the final simulation result.# 最终模拟结果
        print(alpha_id)
    except:
        print("no location, sleep for 10 seconds and try next alpha.“没有位置，睡10秒然后尝试下一个字母。”")
        sleep(10)

##################测试前两个##############################
from time import sleep

for index,alpha in enumerate(alpha_list[0:2],start=1):
    sim_resp = sess.post(
        'https://api.worldquantbrain.com/simulations',
        json=alpha,
    )

    try:
        sim_progress_url = sim_resp.headers['Location']
        while True:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = float(sim_progress_resp.headers.get("Retry-After", 0))
            if retry_after_sec == 0:  # simulation done!模拟完成!
                break
            sleep(retry_after_sec)
        alpha_id = sim_progress_resp.json()["alpha"]  # the final simulation result.# 最终模拟结果
        print(alpha_id)
    except:
        print("no location, sleep for 10 seconds and try next alpha.“没有位置，睡10秒然后尝试下一个字母。”")
        sleep(10)

#################################################


xYrZ15q
L9MQKN6
dOWGYMv
kVNA9dO
A0VA22g
Ev69ZaL
mgoEp9K
KOJW0a8
9VxNLw1
WPOvK0P
1xNr1m6
r5RZz78
j3Yq9kE
V6xAdPM
xYrJrZn
83RMNEV
l8Aog5x
Y51R816
Jxe6zMn
kVNz9l8
L9M0YLa
M1YJvnk
gQenlll
WPOvvWN
zYdvvY1
2O51r6w
vkAK8Qa
no location, sleep for 10 seconds and try next alpha.“没有位置，睡10秒然后尝试下一个字母。”
gQaYvgO
Jx8VRkm
Y5ajwe6
L9KP61m
V6NvYoA
o6a1KWm
M16jQY8
7ZYknlL
nKa1KNE
l8aLg9N
r571NwJ
KOgKQJl
83Nm0n7
o6amm7v
Z0aYeZ1
Y5aPdO6
1xWYVV6
j3a2wgo
zYdqJRo
KOgPXgl
WPljN3x
390qzmN
bRalqZm
wYopEQp
V6NkdXb
390q0Ee
5QJ8G16
GdmrKwP
dOa5zvx
o6amkQ2
2O581RJ
83N5bNW
xYvO6Zw
V6N1gP0
mgawOlE
EvQO06R
OrWQbKR
6rMmpYK
1xWkwzW
gQakQnM
kVaEggP
e9ak57N
aLajQKx
qW793WO
Q30Z6z5
6rMmQX7
83N5xKo
WPl1rRQ
Y5aLRqv
OrWOQ31
390wLVN
xYvqMKp
OrWOwvJ
EvQ1bWR
wYoqeqQ
EvQ1el0
xYvqrkg
EvQ1mEm
7ZYW3R5
M16Ozdo
PZ5NAEW
A0x983g
PZ5NkYx
qW71MmE
7ZYAW9O
RVe3Wjo
OrWYaL7
9VlvoZV
9Vlvgo2
Q30J5VW
j3aE02Z
vkAqNzA
NQPw7Gp
o6apLPE
e9ap66g
r5738pE
Z0a515d
390bveX
9VlveKd
WPlq53O
nKaR1ql
KOgdlEE
XWaQqlz
390rgoe
GdmAYVJ
EvQWbjm
no location, sleep fo

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))